# using vegan to map out stats and ordination plots for 16S sequences

In [ ]:
ps<-readRDS(file = "/work/pi_sarah_gignouxwolfsohn_uml_edu/caroline/BEL_16S_ITS2/BEL_16S_outputs/ps_16S.rds")
#removing any taxa that don't show up in any samples to speed up the process
ps <- prune_taxa(taxa_sums(ps) > 0, ps)
ps

In [ ]:
#normalizing ps by converting rawcounts into relative abundances
#so samples with more reads wont be over represented
#using ps bc only to the count data (OTU table), while preserving the rest of the object
ps_norm = transform_sample_counts(ps, function(x) 1E6 * x / sum(x))

In [ ]:
#isolate just bacteria
ps_norm_bac=subset_taxa(ps_norm, Kingdom=="Bacteria")
#remove chloroplast order
ps_norm_nochlo=subset_taxa(ps_norm_bac, Order!="Chloroplast")
#remove mitochondria family
ps_norm_nomit=subset_taxa(ps_norm_nochlo, Family!="Mitochondria")
ps_norm_nomit

In [ ]:
# convert the sample_data() within a phyloseq object to a vegan compatible data object
pssd2veg <- function(ps_norm_nomit) {
  sd_nomit <- sample_data(ps_norm_nomit)
  return(as(sd_nomit,"data.frame"))
}
#using phyloseq nmds plot no chloroplast
sample_nomit <- pssd2veg(ps_norm_nomit)

In [ ]:
# convert the otu_table() within a phyloseq object to a vegan compatible data object
psotu2veg <- function(ps_norm_nochlo) {
  otu_nochlo <- otu_table(ps_norm_nochlo)
  if (taxa_are_rows(otu_nochlo)) {
    otu_nochlo <- t(otu_nochlo)
  }
  return(as(otu_nochlo, "matrix"))
}

# Extract normalized OTU matrix and sample data
otu_nochlo <- psotu2veg(ps_norm_nochlo)

### clean sample metadata

In [ ]:
sample_nomit <- as.data.frame(sample_data(ps_norm_nomit))
#save sammple names as a column so tidy doesn't get rid of it during filtering
sample_nomit$SampleID <- rownames(sample_nomit)

In [ ]:
#cleaning up sample_norm to only be metadata
sample_clean <- sample_nochlo[, c("Health_Status", "colony", "Date_16S", "double_band", "transect", "species", "MonthYear", "Condition")]

In [ ]:
## check to make sure meta and otu table are still compatible

# Should return TRUE
all(rownames(sample_clean) == rownames(otu_nochlo))

# vegan cluster analysis

In [ ]:
# merge otu and sam_clean by sample ID
otu_nochlo$SampleID <- rownames(otu_nochlo)
sample_clean$SampleID <- rownames(sample_clean)

In [ ]:
otu_merged <- merge(otu_nochlo, sample_clean, by = "SampleID",
                    all = TRUE, sort = FALSE)
rownames(otu_merged) <- otu_merged$SampleID
otu_merged$SampleID <- NULL
otu_nochlo$SampleID <- NULL

In [ ]:
class(otu_merged)
head(otu_merged)
nrow(otu_nochlo)
nrow(otu_merged)
all(rownames(otu_nochlo) == rownames(otu_merged))

## PC clustering of host species with hellinger distance

In [ ]:
# Hellinger distance, comparable to euclidean 
ord <- decostand(otu_nochlo, method = "hellinger")

In [ ]:
str(otu_merged$Species)
table(otu_merged$Species)

In [ ]:
# plot settings
options(repr.plot.width=20, repr.plot.height=18)

In [ ]:
#by species 
disp <- "sites" 
scl <- "symmetric" 

#transforming species col from chr to factor 
otu_merged$species <- factor(otu_merged$species)
# PCA via rda()
pca_mod <- rda(ord)
#color points
# Color vector
col_vec <-c("red", "blue", "orange", "grey", "purple", "green")
cols <- col_vec[otu_merged$species]
plot(pca_mod, type = "n", scaling = scl, display = disp) 
ordihull(pca_mod, groups = otu_merged$species, col = col_vec, scaling = scl, lwd = 2) 
ordispider(pca_mod, groups = otu_merged$species, col = col_vec, scaling = scl, label = TRUE) 
points(pca_mod, display = disp, scaling = scl, pch = 21, col = "red", bg = "yellow")

## dendogram cluster analysis

In [ ]:
dij <- vegdist(otu_nochlo) ## bray curtis dissimilarity
clu <- hclust(dij, method = "average")
# 2 clusters bc I know Date_16S is already driving into 2 clusters
grp <- cutree(clu, 8)

In [ ]:
# visualizing the parent dendogram
plot(clu); rect.hclust(clu, k=8, border="red")

## heatmap with dendogram